In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv(f"{path}/Q1_data.csv")

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
target = "Delivery_Time"

In [ ]:

df.drop(columns=["Order_ID"])

In [ ]:
missing_percentage = (x.isnull().sum() / len(x)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
missing_data.head(10)


df[target].fillna(df[target].median())
df['Weather'].fillna(df['Weather'].mode()[0])
df['Traffic_Level'].fillna(df['Traffic_Level'].mode()[0])
df['Time_of_Day'].fillna(df['Time_of_Day'].mode()[0])
df['Courier_Experience_yrs'].fillna(df['Courier_Experience_yrs'].median())

In [ ]:
duplicate_rows = df.duplicated().sum()
if duplicate_rows > 0:
    print(f"Found {duplicate_rows} duplicate rows. Removing them...")
    df.drop_duplicates(inplace=True)
    print("Duplicate rows removed.")
else:
    print("No duplicate rows found.")

In [ ]:
from sklearn.preprocessing import OneHotEncoder

categorical_columns =["Weather", "Traffic_Level", "Time_of_Day", "Vehicle_Type"]
encoder = OneHotEncoder(handle_unknown = 'error', sparse_output = False, categories='auto')
encoder.fit(df[categorical_columns])
df_encoded = encoder.transform(df[categorical_columns])

In [ ]:
from sklearn.preprocessing import StandardScaler

standard_scaler = StandardScaler()
data_standard_scaled = standard_scaler.fit_transform(df_encoded)

In [ ]:
#test
df.head()

In [ ]:
target = "Delivery_Time"
x = df.drop(columns=[target,"Weather","Traffic_Level", "Time_of_Day", "Vehicle_Type"])
y = df[target]

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

model = RandomForestClassifier(n_estimators = 300, max_depth = 6, random_state = 42)

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []

for train_idx, val_idx in kfold.split(x, y):
    X_fold_train, X_fold_val = x[train_idx], x[val_idx]
    y_fold_train, y_fold_val = y[train_idx], y[val_idx]

    # Train and predict
    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)

    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))

mae_scores = np.array(mae_scores)

print(f"5-Fold CV Results:")
print(f"MAE:  ${mae_scores.mean():,.2f}")

In [ ]:
import matplotlib.pyplot as plt
importance = pd.DataFrame({
    'feature': x,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'], color='purple')
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: